In [ ]:
!pip install -q torch transformers datasets rouge-score matplotlib pillow
!git clone https://github.com/musfiraumar/MAE-VAE-DDPM-RAG
%cd MAE-VAE-DDPM-RAG

from google.colab import drive
drive.mount('/content/drive')

fatal: destination path 'MAE-VAE-DDPM-RAG' already exists and is not an empty directory.
/content/MAE-VAE-DDPM-RAG
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/MAE-VAE-DDPM-RAG/part_d_rag

/content/MAE-VAE-DDPM-RAG/part_d_rag


In [ ]:
%%writefile chunking.py
import numpy as np
def chunk_words(text,max_words=150,overlap=30):
    words=text.split()
    if not words:
        return []
    step=max_words-overlap
    chunks=[]
    start=0
    while True:
        piece=words[start:start+max_words]
        chunks.append(" ".join(piece))
        if start+max_words>=len(words):
            break
        start+=step
    return chunks
def build_chunks(corpus):
    texts,doc_ids=[],[]
    for doc_id,text in corpus.items():
        for c in chunk_words(text):
            texts.append(c)
            doc_ids.append(doc_id)
    return texts,np.array(doc_ids,dtype=np.int64)

Writing chunking.py


In [ ]:
%%writefile embedding.py
import torch
from transformers import AutoTokenizer,AutoModel
ENCODER_NAME="sentence-transformers/all-MiniLM-L6-v2"
def load_encoder(device):
    tok=AutoTokenizer.from_pretrained(ENCODER_NAME)
    enc=AutoModel.from_pretrained(ENCODER_NAME).to(device).eval()
    return tok,enc
@torch.no_grad()
def embed_texts(texts,tok,enc,device,batch_size=128,max_length=256):
    out=[]
    for i in range(0,len(texts),batch_size):
        batch=texts[i:i+batch_size]
        t=tok(batch,padding=True,truncation=True,max_length=max_length,return_tensors="pt").to(device)
        hidden=enc(**t).last_hidden_state
        mask=t["attention_mask"].unsqueeze(-1).float()
        mean=(hidden*mask).sum(dim=1)/mask.sum(dim=1).clamp(min=1e-9)
        mean=mean/mean.norm(dim=1,keepdim=True)
        out.append(mean.cpu())
        if (i//batch_size)%50==0:
            print(f"  embedded {min(i+batch_size,len(texts))}/{len(texts)}")
    return torch.cat(out).numpy()

Writing embedding.py


In [ ]:
%%writefile build_index.py
import sys,os,json,time
import numpy as np
import torch
sys.path.append("../starter")
from bioasq import load_corpus
from chunking import build_chunks
from embedding import load_encoder,embed_texts
device="cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("../results/index",exist_ok=True)
corpus=load_corpus()
texts,doc_ids=build_chunks(corpus)
print("abstracts:",len(corpus))
print("chunks   :",len(texts))
tok,enc=load_encoder(device)
t0=time.time()
emb=embed_texts(texts,tok,enc,device)
secs=time.time()-t0
print("embedding matrix:",emb.shape,emb.dtype,f"({secs:.0f}s)")
np.save("../results/index/chunk_embeddings.npy",emb)
json.dump({"abstracts":len(corpus),"chunks":len(texts),"embed_seconds":secs},open("../results/index/index_info.json","w"))

Writing build_index.py


In [ ]:
%%writefile search.py
import numpy as np
class Index:
    def __init__(self,emb,chunk_doc_ids):
        self.emb=emb
        self.chunk_doc_ids=chunk_doc_ids
        change=np.r_[True,chunk_doc_ids[1:]!=chunk_doc_ids[:-1]]
        self.starts=np.flatnonzero(change)
        self.doc_ids=chunk_doc_ids[self.starts]
    def search_docs(self,q_vecs,k=20):
        scores=q_vecs@self.emb.T
        doc_scores=np.maximum.reduceat(scores,self.starts,axis=1)
        order=np.argsort(-doc_scores,axis=1)[:,:k]
        return self.doc_ids[order],np.take_along_axis(doc_scores,order,axis=1)
    def search_chunks(self,q_vec,k=5):
        scores=self.emb@q_vec
        order=np.argsort(-scores)[:k]
        return order,scores[order]

Writing search.py


In [ ]:
%%writefile metrics.py
import numpy as np
def hit_at_k(ranked,relevant,k):
    return 1.0 if len(set(ranked[:k])&relevant)>=1 else 0.0
def recall_at_k(ranked,relevant,k):
    return len(set(ranked[:k])&relevant)/min(k,len(relevant))
def mrr_at_10(ranked,relevant):
    for rank,doc in enumerate(ranked[:10],start=1):
        if doc in relevant:
            return 1.0/rank
    return 0.0
def retrieval_metrics(rankings,relevants,ks=(1,5,10)):
    out={}
    for k in ks:
        out[f"Hit@{k}"]=float(np.mean([hit_at_k(r,rel,k) for r,rel in zip(rankings,relevants)]))
        out[f"Recall@{k}"]=float(np.mean([recall_at_k(r,rel,k) for r,rel in zip(rankings,relevants)]))
    out["MRR@10"]=float(np.mean([mrr_at_10(r,rel) for r,rel in zip(rankings,relevants)]))
    return out

Writing metrics.py


In [ ]:
%%writefile evaluate_retrieval.py
import sys,json
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.append("../starter")
from bioasq import load_corpus,load_questions
from chunking import build_chunks
from embedding import load_encoder,embed_texts
from search import Index
from metrics import retrieval_metrics,recall_at_k,hit_at_k
device="cuda" if torch.cuda.is_available() else "cpu"
corpus=load_corpus()
qs=load_questions("test",corpus)
texts,chunk_doc_ids=build_chunks(corpus)
emb=np.load("../results/index/chunk_embeddings.npy")
assert emb.shape[0]==len(texts),"index does not match the chunks: rebuild it"
index=Index(emb,chunk_doc_ids)
print("abstracts:",len(corpus),"| chunks:",len(texts),"| test questions:",len(qs))
print("mean relevant abstracts per question:",round(float(np.mean([len(q["relevant_ids"]) for q in qs])),3))
tok,enc=load_encoder(device)
q_vecs=embed_texts([q["question"] for q in qs],tok,enc,device)
rankings=[]
for i in range(0,len(qs),64):
    ids,_=index.search_docs(q_vecs[i:i+64],k=20)
    rankings.extend(ids.tolist())
relevants=[set(q["relevant_ids"]) for q in qs]
table4=retrieval_metrics(rankings,relevants,ks=(1,5,10))
for name,value in table4.items():
    print(f"{name:10s} {value:.4f}")
ks=list(range(1,21))
recall_curve=[float(np.mean([recall_at_k(r,rel,k) for r,rel in zip(rankings,relevants)])) for k in ks]
plt.figure(figsize=(6,4))
plt.plot(ks,recall_curve,marker="o")
plt.xticks(ks[::2])
plt.ylim(0,1)
plt.xlabel("k")
plt.ylabel("Recall@k")
plt.title("Retrieval: Recall@k on BioASQ test questions")
plt.grid(alpha=0.3)
plt.savefig("../results/rag_recall_at_k.png",dpi=120,bbox_inches="tight")
per_question=[{"qid":q["qid"],"top20":r,"hit10":hit_at_k(r,rel,10)} for q,r,rel in zip(qs,rankings,relevants)]
json.dump({"table4":table4,"recall_curve":recall_curve,"per_question":per_question},open("../results/retrieval_results.json","w"))
print("questions with Hit@10 = 0:",sum(1 for p in per_question if p["hit10"]==0))

Writing evaluate_retrieval.py


In [ ]:
%%writefile generation.py
import re
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM
GEN_NAME="Qwen/Qwen2.5-1.5B-Instruct"
NOTICE="For educational use only. Not medical advice."
IDK_SENTENCE="I don't know based on the provided passages."
SYSTEM_PROMPT=("You are a careful biomedical research assistant. Answer the question using ONLY the "+"numbered passages provided. After each statement, cite the passage number(s) that support "+"it in square brackets, for example [1] or [2][4]. Do not use any knowledge from outside the "+"passages. If the passages do not contain the answer, reply exactly: "+IDK_SENTENCE+" Keep the answer short and factual.")
IDK_PATTERNS=["i don't know","i do not know","do not contain","does not contain","doesn't contain","don't contain","not contain the answer","cannot be answered","no information"]
def load_generator(device):
    tok=AutoTokenizer.from_pretrained(GEN_NAME)
    dtype=torch.float16 if device=="cuda" else torch.float32
    model=AutoModelForCausalLM.from_pretrained(GEN_NAME,torch_dtype=dtype).to(device).eval()
    return tok,model
def build_messages(question,passages):
    context="\n\n".join(f"[{i+1}] {p}" for i,p in enumerate(passages))
    user=f"Passages:\n{context}\n\nQuestion: {question}"
    return [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user}]
@torch.no_grad()
def answer_question(question,passages,tok,model,device):
    prompt=tok.apply_chat_template(build_messages(question,passages),tokenize=False,add_generation_prompt=True)
    inputs=tok(prompt,return_tensors="pt").to(device)
    out=model.generate(**inputs,max_new_tokens=256,do_sample=False,temperature=None,top_p=None,top_k=None,repetition_penalty=1.0)
    new_tokens=out[0,inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens,skip_special_tokens=True).strip()
def cited_numbers(answer):
    nums=[]
    for group in re.findall(r"\[([\d,\s]+)\]",answer):
        for part in group.split(","):
            if part.strip().isdigit():
                nums.append(int(part))
    return nums
def said_idk(answer):
    a=answer.lower().replace("’","'")
    return any(p in a for p in IDK_PATTERNS)

Writing generation.py


In [ ]:
%%writefile run_generation.py
import sys,json
import numpy as np
import torch
sys.path.append("../starter")
from bioasq import load_corpus,load_questions
from chunking import build_chunks
from embedding import load_encoder,embed_texts
from search import Index
from generation import load_generator,answer_question,cited_numbers,said_idk
device="cuda" if torch.cuda.is_available() else "cpu"
N_QUESTIONS,TOP_CHUNKS=100,5
corpus=load_corpus()
qs=load_questions("test",corpus)[:N_QUESTIONS]
texts,chunk_doc_ids=build_chunks(corpus)
emb=np.load("../results/index/chunk_embeddings.npy")
assert emb.shape[0]==len(texts)
index=Index(emb,chunk_doc_ids)
enc_tok,enc=load_encoder(device)
gen_tok,gen=load_generator(device)
with open("../results/rag_answers.jsonl","w") as f:
    for n,q in enumerate(qs,start=1):
        q_vec=embed_texts([q["question"]],enc_tok,enc,device)[0]
        idx,scores=index.search_chunks(q_vec,k=TOP_CHUNKS)
        retrieved=[{"passage_no":r+1,"chunk_index":int(i),"abstract_id":int(chunk_doc_ids[i]),"score":float(s),"text":texts[i]} for r,(i,s) in enumerate(zip(idx,scores))]
        answer=answer_question(q["question"],[p["text"] for p in retrieved],gen_tok,gen,device)
        rec={"qid":q["qid"],"question":q["question"],"reference_answer":q["answer"],"relevant_ids":q["relevant_ids"],"retrieved":retrieved,"answer":answer,"cited":cited_numbers(answer),"idk":said_idk(answer)}
        f.write(json.dumps(rec)+"\n")
        f.flush()
        print(f"[{n}/{N_QUESTIONS}] {q['question'][:70]}")

Writing run_generation.py


In [ ]:
%%writefile evaluate_generation.py
import json,re
from rouge_score import rouge_scorer
recs=[json.loads(line) for line in open("../results/rag_answers.jsonl")]
scorer=rouge_scorer.RougeScorer(["rougeL"],use_stemmer=True)
scores=[]
for r in recs:
    clean=re.sub(r"\[[\d,\s]+\]","",r["answer"])
    scores.append(scorer.score(r["reference_answer"],clean)["rougeL"].fmeasure)
rouge_l=sum(scores)/len(scores)
correct=total=no_cite=0
for r in recs:
    relevant=set(r["relevant_ids"])
    nums=set(r["cited"])
    if not nums:
        no_cite+=1
    for n in nums:
        total+=1
        if 1<=n<=len(r["retrieved"]) and r["retrieved"][n-1]["abstract_id"] in relevant:
            correct+=1
precision=correct/total if total else float("nan")
idk_rate=sum(r["idk"] for r in recs)/len(recs)
print(f"ROUGE-L F             : {rouge_l:.4f}")
print(f"citation precision    : {precision:.4f}  ({correct}/{total} cited passages)")
print(f"'I don't know' rate   : {idk_rate:.2%}  ({sum(r['idk'] for r in recs)}/{len(recs)})")
print(f"answers with no citation: {no_cite}/{len(recs)}")
json.dump({"rougeL":rouge_l,"citation_precision":precision,"idk_rate":idk_rate,"no_citation_answers":no_cite},open("../results/generation_metrics.json","w"),indent=2)

Writing evaluate_generation.py


In [ ]:
%%writefile human_eval_rag.py
import sys,csv,json,random,os
SEED=2026
recs=[json.loads(line) for line in open("../results/rag_answers.jsonl")]
picked=sorted(random.Random(SEED).sample(range(len(recs)),20))
mode=sys.argv[1]
if mode=="make":
    with open("../results/rag_human_eval_sheet.md","w") as f:
        for n,i in enumerate(picked,start=1):
            r=recs[i]
            f.write(f"## Item {n}  (qid {r['qid']})\n\n**Question:** {r['question']}\n\n")
            for p in r["retrieved"]:
                f.write(f"[{p['passage_no']}] (abstract {p['abstract_id']}) {p['text']}\n\n")
            f.write(f"**Model answer:** {r['answer']}\n\n**Reference answer:** {r['reference_answer']}\n\n---\n\n")
    for m in (1,2):
        with open(f"../results/rag_marks_member{m}.csv","w",newline="") as f:
            w=csv.writer(f)
            w.writerow(["qid","supported","correct"])
            for i in picked:
                w.writerow([recs[i]["qid"],"",""])
    print("sheet written; each member fills supported/correct with 1 or 0 in their own CSV")
elif mode=="merge":
    def read(path):
        return {int(r["qid"]):r for r in csv.DictReader(open(path))}
    m1,m2=read("../results/rag_marks_member1.csv"),read("../results/rag_marks_member2.csv")
    path="../results/human_eval.csv"
    rows=list(csv.DictReader(open(path))) if os.path.exists(path) else []
    for r in rows:
        r.setdefault("part","C")
    rows=[r for r in rows if r["part"]!="D"]
    for i in picked:
        q=recs[i]["qid"]
        rows.append({"part":"D","qid":q,"supported_member1":m1[q]["supported"],"supported_member2":m2[q]["supported"],"correct_member1":m1[q]["correct"],"correct_member2":m2[q]["correct"]})
    fields=["part","id","true_class","member1_guess","member2_guess","member1_correct","member2_correct","agree","qid","supported_member1","supported_member2","correct_member1","correct_member2"]
    with open(path,"w",newline="") as f:
        w=csv.DictWriter(f,fieldnames=fields,restval="")
        w.writeheader()
        w.writerows(rows)
    d=[r for r in rows if r["part"]=="D"]
    for key in ("supported_member1","supported_member2","correct_member1","correct_member2"):
        print(f"{key}: {sum(int(r[key]) for r in d if r[key].strip()!='')}/20")

Overwriting human_eval_rag.py


In [ ]:
%%writefile checks.py
import sys
import numpy as np
import torch
sys.path.append("../starter")
from bioasq import load_corpus,load_questions
from chunking import build_chunks,chunk_words
from embedding import load_encoder,embed_texts
from search import Index
from metrics import retrieval_metrics
device="cuda" if torch.cuda.is_available() else "cpu"
corpus=load_corpus()
texts,doc_ids=build_chunks(corpus)
longest=max(len(t.split()) for t in texts)
print("longest chunk (words):",longest)
assert longest<=150
pairs=bad=0
for text in corpus.values():
    ch=chunk_words(text)
    for a,b in zip(ch[:-1],ch[1:]):
        pairs+=1
        bad+=a.split()[-30:]!=b.split()[:30]
print(f"overlap check: {pairs} consecutive pairs, {bad} violations")
assert bad==0
index_emb=np.load("../results/index/chunk_embeddings.npy")
assert index_emb.shape[0]==len(texts)
index=Index(index_emb,doc_ids)
assert len(index.doc_ids)==len(set(doc_ids.tolist()))
print("chunks per abstract are contiguous: OK")
norms=np.linalg.norm(index_emb,axis=1)
print(f"embedding norms: min {norms.min():.6f}  max {norms.max():.6f}")
assert np.allclose(norms,1.0,atol=1e-4)
tok,enc=load_encoder(device)
short="Aspirin inhibits platelet aggregation."
alone=embed_texts([short],tok,enc,device)[0]
padded=embed_texts([short,texts[0]],tok,enc,device)[0]
print("cosine(alone, padded):",float(alone@padded))
assert float(alone@padded)>0.9999
lens=[len(x) for x in tok(texts[:2000])["input_ids"]]
print(f"chunks longer than 256 tokens (first 2000): {np.mean(np.array(lens)>256):.1%}")
qs=load_questions("test",corpus)
other=list(corpus.keys())[:50]
rankings,relevants=[],[]
for q in qs:
    rel=set(q["relevant_ids"])
    rankings.append(sorted(rel)+[d for d in other if d not in rel])
    relevants.append(rel)
perfect=retrieval_metrics(rankings,relevants,ks=(1,5,10))
print("perfect retriever:",perfect)
assert all(abs(v-1.0)<1e-9 for v in perfect.values())

Writing checks.py


In [ ]:
%%writefile failure_analysis.py
import sys,json,random
import numpy as np
import torch
sys.path.append("../starter")
from bioasq import load_corpus,load_questions
from chunking import build_chunks
from embedding import load_encoder,embed_texts
from search import Index
from generation import load_generator,answer_question,cited_numbers,said_idk
device="cuda" if torch.cuda.is_available() else "cpu"
corpus=load_corpus()
qs=load_questions("test",corpus)
texts,chunk_doc_ids=build_chunks(corpus)
index=Index(np.load("../results/index/chunk_embeddings.npy"),chunk_doc_ids)
res=json.load(open("../results/retrieval_results.json"))["per_question"]
assert [r["qid"] for r in res]==[q["qid"] for q in qs]
fails=[i for i,r in enumerate(res) if not (set(r["top20"][:10])&set(qs[i]["relevant_ids"]))]
print(len(fails),"of",len(qs),"questions failed (no relevant abstract in the top 10)")
first,rest=[i for i in fails if i<100],[i for i in fails if i>=100]
rng=random.Random(0)
rng.shuffle(first)
rng.shuffle(rest)
chosen=(first+rest)[:3]
enc_tok,enc=load_encoder(device)
gen_tok,gen=load_generator(device)
def short(t,n=60):
    return " ".join(t.split()[:n])+" ..."
with open("../results/failure_cases.md","w") as f:
    for i in chosen:
        q=qs[i]
        qv=embed_texts([q["question"]],enc_tok,enc,device)
        ids,sc=index.search_docs(qv,k=len(index.doc_ids))
        ids,sc=ids[0].tolist(),sc[0]
        pos={d:r for r,d in enumerate(ids,start=1)}
        best_rank=min(pos[d] for d in q["relevant_ids"])
        idx,_=index.search_chunks(qv[0],k=5)
        passages=[texts[j] for j in idx]
        ans=answer_question(q["question"],passages,gen_tok,gen,device)
        f.write(f"## Question (test index {i}, qid {q['qid']})\n{q['question']}\n\n")
        f.write(f"**Reference answer:** {q['answer']}\n\n")
        f.write(f"**Best rank of a relevant abstract:** {best_rank} of {len(ids)}\n\n")
        for d in q["relevant_ids"][:2]:
            f.write(f"- relevant abstract {d} (rank {pos[d]}): {short(corpus[d])}\n")
        f.write("\n**Top 3 retrieved:**\n")
        for r in range(3):
            f.write(f"- abstract {ids[r]} (score {sc[r]:.3f}): {short(corpus[ids[r]],40)}\n")
        f.write(f"\n**Generator answer:** {ans}\n\ncited: {cited_numbers(ans)}  |  said IDK: {said_idk(ans)}\n\n")
        for k,j in enumerate(idx,start=1):
            f.write(f"- passage [{k}] (abstract {int(chunk_doc_ids[j])}): {short(texts[j],40)}\n")
        f.write("\n---\n\n")
print("written to ../results/failure_cases.md")

Writing failure_analysis.py


In [ ]:
%%writefile make_samples.py
import json,random
recs=[json.loads(line) for line in open("../results/rag_answers.jsonl")]
ret={r["qid"]:r for r in json.load(open("../results/retrieval_results.json"))["per_question"]}
idk=[r for r in recs if r["idk"]]
failed=[r for r in recs if ret[r["qid"]]["hit10"]==0]
print("IDK answers:",len(idk),"| retrieval failures among the 100:",len(failed))
chosen=[]
if idk:
    chosen.append(idk[0])
for r in failed:
    if r not in chosen:
        chosen.append(r)
        break
others=[r for r in recs if r not in chosen]
chosen+=random.Random(0).sample(others,5-len(chosen))
with open("../results/samples.md","w") as f:
    f.write("> For educational use only. Not medical advice.\n\n")
    for n,r in enumerate(chosen,start=1):
        tags=[]
        if ret[r["qid"]]["hit10"]==0:
            tags.append("retrieval failed")
        if r["idk"]:
            tags.append("model said it does not know")
        f.write(f"## Sample {n} {'('+', '.join(tags)+')' if tags else ''}\n\n")
        f.write(f"**Question:** {r['question']}\n\n**Answer:** {r['answer']}\n\n**Cited passages:**\n\n")
        for num in sorted(set(r["cited"])):
            if 1<=num<=len(r["retrieved"]):
                p=r["retrieved"][num-1]
                ok="relevant" if p["abstract_id"] in r["relevant_ids"] else "not relevant"
                f.write(f"- [{num}] abstract {p['abstract_id']} ({ok}): {p['text']}\n")
        f.write(f"\n**Reference answer:** {r['reference_answer']}\n\n---\n\n")

Writing make_samples.py


In [ ]:
cd ..

/content/MAE-VAE-DDPM-RAG


In [ ]:
cd /content/MAE-VAE-DDPM-RAG/part_d_rag

/content/MAE-VAE-DDPM-RAG/part_d_rag


In [ ]:
!python build_index.py

README.md: 100% 1.76k/1.76k [00:00<00:00, 4.24MB/s]

text-corpus/test-00000-of-00001.parquet: downloading bytes:  80% 28.1M/35.3M [00:00<00:00, 53.7MB/s, 1.64MB/s  ]
text-corpus/test-00000-of-00001.parquet: downloading bytes: 100% 35.1M/35.1M [00:00<00:00, 36.1MB/s, 3.39MB/s  ]
text-corpus/test-00000-of-00001.parquet: reconstructing file: 100% 35.3M/35.3M [00:00<00:00, 36.3MB/s, 3.45MB/s  ]
Generating test split: 100% 40181/40181 [00:00<00:00, 119392.57 examples/s]
abstracts: 40181
chunks   : 79332
config.json: 100% 612/612 [00:00<00:00, 2.37MB/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.42MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 13.4MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 34.4MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 450kB/s]

model.safetensors: downloading bytes:  93% 84.4M/90.9M [00:00<00:00, 151MB/s, 6.74MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:00<00:00, 105MB/s, 8.18MB/s  ]
model.safetensors: reconstructing file:

In [ ]:

!python checks.py


longest chunk (words): 150
overlap check: 39151 consecutive pairs, 0 violations
chunks per abstract are contiguous: OK
embedding norms: min 1.000000  max 1.000000
Loading weights: 100% 103/103 [00:00<00:00, 15313.65it/s]
  embedded 1/1
  embedded 2/2
cosine(alone, padded): 1.0
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (584 > 512). Running this sequence through the model will result in indexing errors
chunks longer than 256 tokens (first 2000): 10.3%

question-answer-passages/train-00000-of-(…): downloading bytes:   0% 0.00/1.12M [00:00<?, ?B/s]
question-answer-passages/train-00000-of-(…): downloading bytes: 100% 1.11M/1.11M [00:00<00:00, 1.18MB/s,  109kB/s  ]
question-answer-passages/train-00000-of-(…): reconstructing file: 100% 1.12M/1.12M [00:00<00:00, 1.19MB/s,  110kB/s  ]

question-answer-passages/test-00000-of-0(…): downloading bytes:   0% 0.00/187k [00:00<?, ?B/s]
question-answer-passages/test-00000-of-0(…): d

In [ ]:
!python evaluate_retrieval.py


abstracts: 40181 | chunks: 79332 | test questions: 707
mean relevant abstracts per question: 8.663
Loading weights: 100% 103/103 [00:00<00:00, 4377.08it/s]
  embedded 128/707
Hit@1      0.6223
Recall@1   0.6223
Hit@5      0.8161
Recall@5   0.5863
Hit@10     0.8614
Recall@10  0.6024
MRR@10     0.7061
questions with Hit@10 = 0: 98


In [ ]:
!python run_generation.py


Loading weights: 100% 103/103 [00:00<00:00, 17427.62it/s]
config.json: 100% 660/660 [00:00<00:00, 2.94MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 24.2MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 67.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 94.4MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 104MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 54.6M/3.09G [00:01<00:39, 76.0MB/s, 4.03MB/s  ]
model.safetensors: downloading bytes:   8% 238M/3.09G [00:02<00:12, 234MB/s, 19.6MB/s  ]
model.safetensors: downloading bytes:  16% 479M/3.09G [00:03<00:17, 149MB/s, 38.1MB/s  ]
model.safetensors: reconstructing file:   1% 26.1M/3.09G [00:04<07:19, 6.96MB/s, 1.44MB/s  ]
model.safetensors: reconstructing file:   3% 79.9M/3.09G [00:04<01:37, 30.7MB/s, 2.18MB/s  ]
model.safetensors: downloading bytes:  17% 524M/3.09G [00:04<00:22, 113MB/s

In [ ]:
!python evaluate_generation.py


ROUGE-L F             : 0.1845
citation precision    : 0.7255  (74/102 cited passages)
'I don't know' rate   : 5.00%  (5/100)
answers with no citation: 21/100


In [ ]:
!python human_eval_rag.py make       # members mark separately


sheet written; each member fills supported/correct with 1 or 0 in their own CSV


In [89]:
!python human_eval_rag.py merge


supported_member1: 8/20
supported_member2: 10/20
correct_member1: 11/20
correct_member2: 11/20


In [90]:
!python failure_analysis.py


98 of 707 questions failed (no relevant abstract in the top 10)
Loading weights: 100% 103/103 [00:00<00:00, 811.35it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:08<00:00, 42.17it/s]
  embedded 1/1
  embedded 1/1
  embedded 1/1
written to ../results/failure_cases.md


In [ ]:
!python make_samples.py

IDK answers: 5 | retrieval failures among the 100: 14
